# Evaluating Model Outputs: Accuracy, Precision, Recall, F1 & the Confusion Matrix

This is the reusable evaluation template for the course. Point it at any set of **true labels** and **predicted labels** and it will tell you not just *whether the model ran*, but *whether it was right* — and, critically, *how* it was wrong.

**Time**: ~20 minutes
**Cost**: A few cents at most (we'll estimate before we run anything — see Session 1's token cost guide)

> 📝 **For Juliana**: This notebook has the code and light framing in place. The sections marked **📝 Juliana: expand this** are where the deeper logic explanations and business-decision framing should go — feel free to rewrite the surrounding text too, this is just a starting draft.

## The Model Ran vs. The Model Was Right

An API call can succeed — no error, a clean response, a confident-sounding answer — and still be **wrong**. "It ran" tells you the plumbing worked. It tells you nothing about whether you can trust the output for a business decision.

Evaluation is how you close that gap. Instead of eyeballing a handful of outputs and calling it good, you compare the model's predictions against a set of **known correct answers** (called *ground truth* or *true labels*) and measure exactly how often, and in what way, it gets things wrong.

> 📝 **Juliana: expand this** — a concrete example of "ran but wrong" that would land with agency/consultant folks (e.g. a classifier confidently mislabeling a churn-risk complaint as "general feedback"), and why that distinction matters before a model's output touches a real business decision. This also sets up the Human-in-the-Loop discussion later in the session.

## Setup

If you completed the Session 1 setup guide, your Gemini API key is already saved in Colab Secrets — there's nothing extra to do here. If you haven't, go do that first: [session_1/setup_guide.ipynb](../session_1/setup_guide.ipynb).

In [ ]:
!pip install -q google-genai

In [ ]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
print("Connected.")

## The Evaluation Dataset

To keep this notebook self-contained and privacy-safe, we're using a small **synthetic** dataset — 45 invented customer feedback snippets, each hand-labeled with one of five topics an agency or CX team would recognize. No real customer data, nothing to anonymize.

**Swap this out** for your own labeled data later — anything with a text column and a true-label column works with the rest of this notebook.

In [ ]:
import pandas as pd

CATEGORIES = [
    "Shipping & Delivery",
    "Product Quality",
    "Pricing & Billing",
    "Customer Service",
    "Returns & Refunds",
]

data = [
    ("My package arrived five days later than the estimated delivery date.", "Shipping & Delivery"),
    ("Tracking said it was out for delivery but it never showed up.", "Shipping & Delivery"),
    ("Shipping was actually really fast, got it in two days!", "Shipping & Delivery"),
    ("The box arrived completely crushed, I think it was thrown around.", "Shipping & Delivery"),
    ("Delivery driver left it in the rain even though I asked for it to go behind the gate.", "Shipping & Delivery"),
    ("I paid for express shipping but it still took a week.", "Shipping & Delivery"),
    ("Great communication throughout, I always knew where my order was.", "Shipping & Delivery"),
    ("Wrong address on the label caused a huge delay.", "Shipping & Delivery"),
    ("The courier was really friendly and delivered right on time.", "Shipping & Delivery"),

    ("The fabric started pilling after just one wash.", "Product Quality"),
    ("Way better quality than I expected for the price.", "Product Quality"),
    ("The zipper broke on the second use.", "Product Quality"),
    ("Solid build, feels like it'll last for years.", "Product Quality"),
    ("Colors looked nothing like the photos online.", "Product Quality"),
    ("Stitching came undone within a week.", "Product Quality"),
    ("This is by far the best version of this product I've owned.", "Product Quality"),
    ("The material feels cheap and flimsy.", "Product Quality"),
    ("Exceeded my expectations, very well made.", "Product Quality"),

    ("I was charged twice for the same order.", "Pricing & Billing"),
    ("The price displayed at checkout didn't match what was on my card statement.", "Pricing & Billing"),
    ("Great value for the price, would buy again.", "Pricing & Billing"),
    ("Subscription renewed without any warning and charged my card.", "Pricing & Billing"),
    ("There was a hidden fee I wasn't told about until checkout.", "Pricing & Billing"),
    ("Prices are way too high compared to competitors.", "Pricing & Billing"),
    ("Refund for the price difference was processed quickly.", "Pricing & Billing"),
    ("I appreciate the transparent pricing, no surprises.", "Pricing & Billing"),
    ("My discount code didn't apply and I paid full price.", "Pricing & Billing"),

    ("The support agent was incredibly patient and solved my issue in minutes.", "Customer Service"),
    ("I waited on hold for over an hour and never got through.", "Customer Service"),
    ("Nobody responded to my email for a week.", "Customer Service"),
    ("The chat agent was rude and dismissive.", "Customer Service"),
    ("They went above and beyond to fix my problem.", "Customer Service"),
    ("I had to explain my issue three times to three different agents.", "Customer Service"),
    ("Quick, friendly, and knowledgeable support team.", "Customer Service"),
    ("Support kept transferring me in circles.", "Customer Service"),
    ("Really appreciated the follow-up call to make sure everything was resolved.", "Customer Service"),

    ("My return was processed within two days, very smooth.", "Returns & Refunds"),
    ("I've been waiting three weeks for my refund and still nothing.", "Returns & Refunds"),
    ("The return label they sent didn't work at the post office.", "Returns & Refunds"),
    ("Easiest return process I've ever dealt with.", "Returns & Refunds"),
    ("They refused to refund me even though the item was defective.", "Returns & Refunds"),
    ("Refund showed up on my card exactly as promised.", "Returns & Refunds"),
    ("Had to pay for return shipping even though it was their mistake.", "Returns & Refunds"),
    ("Customer service made the return painless.", "Returns & Refunds"),
    ("Still waiting on a refund from a return I sent back a month ago.", "Returns & Refunds"),
]

df = pd.DataFrame(data, columns=["text", "true_label"])
print(f"{len(df)} labeled examples across {df['true_label'].nunique()} categories")
df.sample(5, random_state=1)

## Generate Predictions

Now let's have Gemini classify each snippet, without telling it the true label, so we have something to evaluate. This is the same pattern as item 4's "first AI assistant use case" — classification — just on our placeholder dataset instead of a live one.

**Before running a batch job, estimate the cost** (habit from the token cost guide). 45 short classification calls on the free tier costs $0 — but it does use ~45 requests of your daily quota, so this is a good moment to check that habit even when the dollar cost is zero.

In [ ]:
CATEGORY_LIST = ", ".join(CATEGORIES)

def classify(text: str) -> str:
    prompt = f"""Classify the following customer feedback into exactly ONE of these categories:
{CATEGORY_LIST}

Feedback: "{text}"

Respond with only the category name, exactly as written above. Nothing else."""

    response = client.models.generate_content(
        model='gemini-2.5-flash-lite',
        contents=prompt,
    )
    label = response.text.strip()
    # Guard against the model returning something slightly off-format
    return label if label in CATEGORIES else "UNKNOWN"

predictions = [classify(text) for text in df["text"]]
df["predicted_label"] = predictions

unknown_count = (df["predicted_label"] == "UNKNOWN").sum()
if unknown_count:
    print(f"⚠️ {unknown_count} response(s) didn't match a known category exactly — worth showing the class as a real-world formatting failure mode.")
print("Done.")
df.head()

## Accuracy

**Accuracy** answers one question: *of everything the model labeled, what percentage did it get exactly right?*

It's the simplest metric — and the easiest to misread. A model can score 90% accuracy and still be useless if the 10% it gets wrong are the cases that matter most to your business (e.g. always missing the angriest, highest-churn-risk complaints).

> 📝 **Juliana: expand this** — why accuracy alone can be misleading, ideally with a marketing example (e.g. an imbalanced dataset where 90% of feedback is "Shipping & Delivery" — a model that *always* guesses that category looks 90% accurate while being useless).

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(df["true_label"], df["predicted_label"])
print(f"Overall accuracy: {accuracy:.1%}  ({int(accuracy * len(df))} of {len(df)} correct)")

## The Confusion Matrix

Accuracy gives you one number. The **confusion matrix** shows you *where the model gets confused* — which categories it mixes up with which. Rows are the true label, columns are what the model predicted; the diagonal is everything it got right, and everything off the diagonal is a specific kind of mistake.

> 📝 **Juliana: expand this** — how to read a confusion matrix in plain language, and why "which mistakes" often matters more to a business than "how many mistakes" (e.g. confusing Pricing & Billing with Customer Service is a much smaller problem than confusing Returns & Refunds with general feedback, if refunds are what trigger a churn-prevention workflow).

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(df["true_label"], df["predicted_label"], labels=CATEGORIES)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CATEGORIES, yticklabels=CATEGORIES)
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.title('Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Precision, Recall & F1 — Per Category

Accuracy treats every category the same. These three metrics look at each category individually, and answer three different questions:

- **Precision** — *of everything the model labeled as this category, how much actually was?* Low precision means the model cries wolf — it over-labels this category.
- **Recall** — *of everything that actually was this category, how much did the model catch?* Low recall means the model misses real cases — it under-labels this category.
- **F1 score** — a single number that balances precision and recall, useful when you want one metric per category instead of two.

> 📝 **Juliana: expand this** — the "which mistake would you rather make" framing per metric. E.g. for Returns & Refunds, missing a real return request (low recall) is usually worse than occasionally flagging something as a return that wasn't (low precision) — because the cost of a false negative and a false positive aren't symmetric. This is a great lead-in to the Human-in-the-Loop discussion: **high-stakes categories need a human check regardless of the model's overall accuracy.**

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(df["true_label"], df["predicted_label"], labels=CATEGORIES, zero_division=0))

## The Reusable Template

Everything above, wrapped into one function. This is the part anyone in the course can copy into their own notebook: swap in your own `true_label` / `predicted_label` columns and run it.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def evaluate(y_true, y_pred, labels=None, title="Evaluation Results"):
    """
    Evaluate classification predictions against ground truth.
    Works on ANY labeled classification task -- not just this dataset.

    y_true, y_pred : lists or pandas Series of labels (same length, same order)
    labels         : ordered list of all possible category names (optional --
                      inferred from the data if not given)
    """
    if labels is None:
        labels = sorted(set(y_true) | set(y_pred))

    accuracy = accuracy_score(y_true, y_pred)
    print(f"{title}")
    print("=" * len(title))
    print(f"Overall accuracy: {accuracy:.1%}\n")

    print("Per-category breakdown (precision / recall / F1):")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted label')
    plt.ylabel('True label')
    plt.title('Confusion Matrix')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    return {"accuracy": accuracy, "confusion_matrix": cm}

# Try it on our dataset:
results = evaluate(df["true_label"], df["predicted_label"], labels=CATEGORIES, title="Topic Classification Evaluation")

## Using This on Your Own Data

To evaluate your own model outputs instead of this placeholder dataset:

1. Get your data into a table with (at minimum) a `true_label` column and a `predicted_label` column — same row order, same category names in both.
2. If you're starting from a CSV: `df = pd.read_csv('your_file.csv')`
3. Run: `evaluate(df['true_label'], df['predicted_label'], labels=[...your categories...])`

That's it — the function doesn't care what the categories are or where the predictions came from (Gemini, another model, or even a human).

## ✅ Wrap-Up

You now have a working, reusable way to answer "was the model actually right?" instead of just "did it run?" — accuracy for the overall picture, the confusion matrix for *where* it gets confused, and precision/recall/F1 for *how* it's wrong on each category.

**Next in Session 2**: Human-in-the-loop validation — at what point does a wrong prediction actually reach a business decision, and where does a human need to check it first? The per-category breakdown above is exactly the tool you'd use to decide that.

**Questions?** Post in the Circle community.